# Federated Unlearning Lab: Colab compute server
Runs this repo's `core/` on a Colab GPU and exposes it through a temporary HTTPS tunnel so your Mac's dashboard can use it as the **Google Colab** backend.

**Before you start:** Runtime > Change runtime type > pick a **GPU**. Then run the cells in order and keep the last cell running.

In [ ]:
!nvidia-smi -L || echo "No GPU: change the runtime type first" 

In [ ]:
import os
if not os.path.exists("federated-unlearning-v2"):
    !git clone -q https://github.com/Unknown183-a/federated-unlearning-v2.git
%cd federated-unlearning-v2
!git pull -q
# Colab already ships a CUDA build of torch/torchvision; this only fills gaps.
!pip install -q -r core/requirements.txt

In [ ]:
import secrets
API_KEY = secrets.token_urlsafe(32)   # new key every session
os.environ["COMPUTE_API_KEY"] = API_KEY
print("Key created (shown again at the end).")

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
import subprocess, sys, time, re
server = subprocess.Popen([sys.executable, "-m", "core.compute.server", "--host", "127.0.0.1", "--port", "8000"],
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(8)
print(open("/content/server.log").read())

In [ ]:
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
t0 = time.time()
while time.time() - t0 < 60 and URL is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m: URL = m.group(0)
print("Tunnel:", URL or "FAILED, re-run this cell")

In [ ]:
print("Put these two lines in  backend/.env  on your Mac, then restart the backend:\n")
print(f"COLAB_BASE_URL={URL}")
print(f"COLAB_API_KEY={API_KEY}")
print("\nThe URL and key change every time this notebook is restarted.")

In [ ]:
# Keep this cell running while you use the dashboard. Stop it to shut the server down.
import time
try:
    while True:
        time.sleep(60)
        if server.poll() is not None:
            print("Server stopped:", open("/content/server.log").read()[-1500:]); break
except KeyboardInterrupt:
    server.terminate(); tunnel.terminate()